> 💻 **This notebook runs locally — embedded mode.** `pip install figuard`; enforcement runs in-process against a local SQLite file (`~/.figuard/figuard.db`). No server, no API key, no network. Same rules as the server, held in lockstep by a shared conformance suite.
>
> **Three ways to run FiGuard — same API, same rules. Swap only the client constructor:**
>
> - 💻 **Embedded** _(this notebook)_ — `FiGuardClient()` — one app/agent, zero infra.
> - ☁️ **Free sandbox** — `FiGuardClient(base_url="https://figuard-sandbox-g1ha.onrender.com", api_key="sb_live_demo")` — quick hosted evaluation (not for production; may be wiped; no SLA).
> - 🏢 **Self-hosted server** — `FiGuardClient(base_url="https://your-figuard", api_key="...")` — budgets shared across many agents/processes; your data, your infra. [Self-host →](https://github.com/figuard/figuard-core#self-hosting)
>
> Fleet features — delegation, category allocations, shadow mode, cross-process budgets — require the sandbox or a server.

In [ ]:
# @title Step 1 — Install and configure FiGuard
import sys, importlib
!pip install "figuard>=0.3.0" --upgrade -q
# Flush any stale cached module from this runtime session
for _mod in list(sys.modules.keys()):
    if _mod.startswith("figuard"):
        del sys.modules[_mod]
import figuard
print(f"✓ FiGuard {figuard.__version__} ready")
import uuid
USER_ID = f"demo_{uuid.uuid4().hex[:8]}"
print(f"Your session ID: {USER_ID}  (labels your budgets locally)")

In [ ]:
from figuard import FiGuardClient

client = FiGuardClient()

budget = client.create_budget(
    user_id=USER_ID,
    total_limit=100_000,
    unit="tokens",
    expires_in="24h",
    authorization_expiry_seconds=300,
    intent_context="LLM research session",
)

print(f"✓ Budget created: {budget.id}")
print(f"  Total limit:   {budget.total_limit:,} tokens")

session_token = budget.session_token

In [ ]:
# Embedded mode has no web UI — get_spend_tree() *is* the spend view (the same data a dashboard renders).
def show_spend_tree(client, budget_id):
    b = client.get_budget(budget_id)
    tree = client.get_spend_tree(budget_id)
    unit = b.currency or b.unit or ""
    print(f"\nSpend tree — {b.quantity_spent}/{b.total_limit} {unit} spent, "
          f"{b.available_quantity} available  ({tree.total_events} events)")
    glyph = {"CONFIRMED": "✓", "AUTHORIZED": "⏳", "DENIED": "✗", "FAILED": "⚠", "VOIDED": "↩"}
    def walk(node, depth=1):
        e = node.event
        amt = e.confirmed_quantity if e.confirmed_quantity is not None else e.requested_quantity
        label = e.description or e.entity_id or e.action_type
        label = f"{label} — " if label else ""
        print("   " * depth + f"{glyph.get(e.decision, '•')} {label}{amt} {e.currency or ''} [{e.decision}]")
        for ch in node.children:
            walk(ch, depth + 1)
    for r in tree.roots:
        walk(r)


In [ ]:
# Authorized LLM call
auth = client.authorize(
    session_token=session_token,
    agent_id="research_agent",
    action_type="LLM_CALL",
    description="GPT-4o: summarize research paper",
    requested_quantity=8_500,
    idempotency_key="llm-call-001",
)
print(f"LLM call 1: {auth.decision} — {auth.approved_quantity:,} tokens")
client.confirm_event(auth.event_id, confirmed_quantity=8_200)
print("✓ Confirmed. 8,200 tokens consumed.")

# Large call — within budget still
auth2 = client.authorize(
    session_token=session_token,
    agent_id="research_agent",
    action_type="LLM_CALL",
    description="Claude 3.5: long-form synthesis",
    requested_quantity=45_000,
    idempotency_key="llm-call-002",
)
print(f"LLM call 2: {auth2.decision} — {auth2.approved_quantity:,} tokens")
client.confirm_event(auth2.event_id, confirmed_quantity=44_100)
print("✓ Confirmed. 44,100 tokens consumed.")

# Over budget
auth3 = client.authorize(
    session_token=session_token,
    agent_id="research_agent",
    action_type="LLM_CALL",
    description="o1: full document re-analysis",
    requested_quantity=60_000,
    idempotency_key="llm-call-003",
)
print(f"LLM call 3: {auth3.decision} — {auth3.denial_reason}")
print("Budget enforced. No tokens consumed.")

show_spend_tree(client, budget.id)